# `assign_formulas` — bind derived / zero / latent variables

Source-agnostic notebook that populates the formula-bound side of `variable_assignments`. Pairs with [assign_eia.ipynb](assign_eia.ipynb), which handles the TS-bound side.

Scope of this first pass: **production-side only** (the 14 physical production nodes, the 4 basin/residual aggregates, the 5 PADD production views, plus `canadian_oil_sands` and its 4 cross-border outflows). Refineries / exports / hubs / pipelines are left for future passes.

Bindings written:

1. **Explicit derived formulas** — basin-state arithmetic (Permian-TX, Bakken-MT, texas_other, montana_other) and explicit rollups (basin aggregates, PADD views, `canadian_oil_sands` sum-over-outflows).
2. **Latent sentinels** — the 2 cross-border pipelines whose per-pipeline split is unobserved (Mainline-CA + Keystone, jointly constrained by `MCRIPP1+2+3 CA2`).
3. **Programmatic zeros** — for each production-class physical node, the structurally-zero non-relational slots (consumption, inventory, balancing_item).
4. **Programmatic single-outflow derivations** — for each production node that has exactly one outflow edge, bind that outflow to the node's production variable (mass balance: `outflow = production` under default zeros).
5. **Programmatic multi-outflow latents** — for production nodes with multiple outflows, mark each per-route flow as `latent()`.

**Skips** any variable that already has a row in `variable_assignments` (i.e. anything bound by `assign_eia` previously). UPSERT semantics — safe to re-run.

In [1]:
import os
from datetime import datetime

import pandas as pd
from sqlalchemy import create_engine, text

PG_URL = (
    f"postgresql+psycopg2://{os.environ.get('PG_USER','eia_user')}:{os.environ.get('PG_PASS','eia_password')}"
    f"@{os.environ.get('PG_HOST','localhost')}:{os.environ.get('PG_PORT','5432')}/{os.environ.get('PG_DB','eia_crude')}"
)
# search_path must include `oil_network` for the variable_assignments CHECK
# trigger to resolve `scenarios` (see assign_eia.ipynb for context).
engine = create_engine(
    PG_URL,
    connect_args={"options": "-csearch_path=oil_network,public"},
    future=True,
)

SCENARIO_ID    = "starter_us_crude_2015_2025"
EFFECTIVE_FROM = "2015-01-01"

def debug(msg):
    print(f"[{datetime.now():%H:%M:%S}] {msg}")

## 1. Explicit derived formulas + latent sentinels

`formula_inputs` lists every other `variable_id` this formula references; `eia:*` tokens in the formula string are TS references resolved at compute time.

In [2]:
EXPLICIT_FORMULAS = [
    # =========================================================================
    # PRODUCTION-SIDE — Tier 1 derivations
    # =========================================================================
    ("production__crude__permian_tx",
     "production__crude__permian - production__crude__permian_nm",
     ["production__crude__permian", "production__crude__permian_nm"],
     "Permian-TX = Permian basin total minus Permian-NM"),
    ("production__crude__bakken_mt",
     "production__crude__bakken - production__crude__bakken_nd",
     ["production__crude__bakken", "production__crude__bakken_nd"],
     "Bakken-MT = Bakken basin total minus Bakken-ND"),
    ("production__crude__texas_other",
     "production__crude__texas_state_view - production__crude__permian_tx - production__crude__eagle_ford_tx",
     ["production__crude__texas_state_view", "production__crude__permian_tx", "production__crude__eagle_ford_tx"],
     "TX residual (East TX + Granite Wash + Barnett) = TX state minus Permian-TX minus Eagle Ford-TX"),
    ("production__crude__montana_other",
     "production__crude__montana_state_view - production__crude__bakken_mt",
     ["production__crude__montana_state_view", "production__crude__bakken_mt"],
     "MT residual (Cedar Creek) = MT state minus Bakken-MT"),

    # PADD residuals (close gap to EIA PADD totals)
    ("production__crude__padd1_other",
     "production__crude__padd1_view",
     ["production__crude__padd1_view"],
     "PADD 1 residual: NY/PA/WV/VA/FL — no explicitly modelled basins, residual = PADD total"),
    ("production__crude__padd2_other",
     ("production__crude__padd2_view "
      "- production__crude__bakken_nd "
      "- production__crude__oklahoma_conventional"),
     ["production__crude__padd2_view",
      "production__crude__bakken_nd", "production__crude__oklahoma_conventional"],
     "PADD 2 residual: KS, IL, IN, KY, MI, MO, NE, OH, SD, TN"),
    ("production__crude__padd3_other",
     ("production__crude__padd3_view "
      "- production__crude__permian_tx "
      "- production__crude__permian_nm "
      "- production__crude__eagle_ford_tx "
      "- production__crude__gulf_of_america "
      "- production__crude__texas_other"),
     ["production__crude__padd3_view",
      "production__crude__permian_tx", "production__crude__permian_nm",
      "production__crude__eagle_ford_tx", "production__crude__gulf_of_america",
      "production__crude__texas_other"],
     "PADD 3 residual: LA onshore, MS, AL, AR"),
    ("production__crude__padd4_other",
     ("production__crude__padd4_view "
      "- production__crude__bakken_mt "
      "- production__crude__wyoming_conventional "
      "- production__crude__colorado_conventional "
      "- production__crude__montana_other"),
     ["production__crude__padd4_view",
      "production__crude__bakken_mt",
      "production__crude__wyoming_conventional", "production__crude__colorado_conventional",
      "production__crude__montana_other"],
     "PADD 4 residual: UT, ID, Niobrara residual"),
    ("production__crude__padd5_other",
     ("production__crude__padd5_view "
      "- production__crude__california_conventional "
      "- production__crude__alaska_north_slope"),
     ["production__crude__padd5_view",
      "production__crude__california_conventional",
      "production__crude__alaska_north_slope"],
     "PADD 5 residual: NV, AZ, OR"),

    # =========================================================================
    # IMPORTS — non-Canadian foreign tanker derivations
    # =========================================================================
    ("production__crude__padd1_imports_agg",
     "0",
     [],
     "Import terminal does not produce crude. Foreign tanker inflow now lives properly in inflow__crude__padd1_imports_agg__foreign_supply."),
    ("production__crude__padd3_imports_agg",
     "0",
     [],
     "Import terminal does not produce crude. Foreign tanker inflow now lives properly in inflow__crude__padd3_imports_agg__foreign_supply."),
    ("production__crude__padd5_imports_agg",
     "0",
     [],
     "Import terminal does not produce crude. Foreign tanker inflow now lives properly in inflow__crude__padd5_imports_agg__foreign_supply."),

    # =========================================================================
    # CANADIAN OIL SANDS aggregate production (foreign source)
    # =========================================================================
    ("production__crude__canadian_oil_sands", "sum",
     ["outflow__crude__canadian_oil_sands__pipe_enbridge_mainline_ca",
      "outflow__crude__canadian_oil_sands__pipe_keystone",
      "outflow__crude__canadian_oil_sands__pipe_trans_mountain_tmx",
      "outflow__crude__canadian_oil_sands__pipe_express_platte"],
     "Canadian oil sands aggregate production = sum of 4 cross-border pipeline outflows (TMX+Express observed; Mainline+Keystone latent)"),

    # =========================================================================
    # EXPORTS — foreign export destination boundary node (added 2026-05-08)
    # =========================================================================
    # foreign_export_destination is the symmetric counterpart of
    # canadian_oil_sands. It receives crude from the 4 PADD-3 export terminals
    # (per-terminal flow latent). Its `consumption` equals the modelled-PADD-3
    # export total (= MCREXP32, mirrored from padd3_exports_view).
    ("production__crude__foreign_export_destination", "0", [],
     "Foreign export destination is a boundary sink — does not produce crude"),
    ("consumption__crude__foreign_export_destination",
     "outflow__crude__padd3_view__foreign_export_destination",
     ["outflow__crude__padd3_view__foreign_export_destination"],
     "Foreign export destination consumption = PADD 3 export total (modelled portion). "
     "Per-terminal split among the 4 PADD-3 export terminals stays latent."),

    # =========================================================================
    # PRODUCTION-SIDE — Tier 2 / aggregate rollups
    # =========================================================================
    ("production__crude__eagle_ford", "sum",
     ["production__crude__eagle_ford_tx"],
     "Eagle Ford basin = single-state Eagle-Ford-TX"),

    # Zeros for view nodes that don't produce/consume crude
    # =========================================================================
    # BALANCING ITEM at region aggregates — now TS-bound via EIA Supply Adjustment.
    # Previously this notebook wrote 6 closure-formula assignments for
    # balancing_item__crude__{usa_view, padd1..5_view} = ΔS − P + C − ΣI + ΣO.
    # As of the 2026-05-12 second binding pass, EIA's published Supply Adjustment
    # series (MCRUA_NUS_2 + MCRUA_R{1..5}0_2) are bound in assign_eia.ipynb as
    # the authoritative observation per Principle 2.8 (exactly one of
    # observed_authoritative or derived holds). The closure relationship is now
    # a Chapter-5 cross-check (computed downstream as a constraint view), not
    # the primary resolver. Do not re-add the tuples below — they will overwrite
    # the TS bindings written by assign_eia.
    # =========================================================================

    ("inflow__crude__padd1_imports_agg__foreign_supply",
     "inflow__crude__padd1_view__foreign_supply",
     ["inflow__crude__padd1_view__foreign_supply"],
     "Foreign tanker inflow at PADD 1 import-terminal aggregate. Mirrors the PADD-view inflow (same edge value, finer-resolution slot)."),
    ("inflow__crude__padd3_imports_agg__foreign_supply",
     "inflow__crude__padd3_view__foreign_supply",
     ["inflow__crude__padd3_view__foreign_supply"],
     "Foreign tanker inflow at PADD 3 import-terminal aggregate. Mirrors the PADD-view inflow (same edge value, finer-resolution slot)."),
    ("inflow__crude__padd5_imports_agg__foreign_supply",
     "inflow__crude__padd5_view__foreign_supply",
     ["inflow__crude__padd5_view__foreign_supply"],
     "Foreign tanker inflow at PADD 5 import-terminal aggregate. Mirrors the PADD-view inflow (same edge value, finer-resolution slot)."),
]

LATENT_VARIABLES = [
    ("outflow__crude__canadian_oil_sands__pipe_enbridge_mainline_ca",
     "Latent: Mainline-CA per-pipeline flow not observed by EIA. Constrained jointly with Keystone by sum(MCRIPP1+2+3 CA2)."),
    ("outflow__crude__canadian_oil_sands__pipe_keystone",
     "Latent: Keystone per-pipeline flow not observed by EIA. Constrained jointly with Mainline-CA by sum(MCRIPP1+2+3 CA2)."),

    # Per-terminal export outflows: jointly constrained by sum = MCREXP32 at
    # foreign_export_destination, but per-terminal split unobserved.
    ("outflow__crude__ingleside_export__foreign_export_destination",
     "Latent: Ingleside per-terminal export volume not published. Jointly constrained with Houston/Nederland/LOOP by MCREXP32."),
    ("outflow__crude__houston_export__foreign_export_destination",
     "Latent: Houston per-terminal export volume not published. Jointly constrained with Ingleside/Nederland/LOOP by MCREXP32."),
    ("outflow__crude__nederland_export__foreign_export_destination",
     "Latent: Nederland per-terminal export volume not published. Jointly constrained with Ingleside/Houston/LOOP by MCREXP32."),
    ("outflow__crude__loop_terminal__foreign_export_destination",
     "Latent: LOOP per-terminal export volume not published. Jointly constrained with Ingleside/Houston/Nederland by MCREXP32."),
]

debug(f"Explicit formulas: {len(EXPLICIT_FORMULAS)}")
debug(f"Latent variables:  {len(LATENT_VARIABLES)}")

[14:24:57] Explicit formulas: 19
[14:24:57] Latent variables:  6


## 2. Programmatic generation

In [3]:
vars_df = pd.read_sql(text("""
    SELECT v.variable_id, v.variable_type, v.node_id, v.related_node_id,
           a.kind, a.node_class, a.node_subtype
    FROM oil_network.variables v
    JOIN oil_network.nodes  n ON n.node_id = v.node_id
    JOIN oil_network.assets a ON a.asset_id = n.asset_id
    WHERE a.node_class = 'production'
       OR a.node_subtype IN ('refinery', 'refining_district_view', 'refining_centre_view',
                             'spr_site',
                             'state_view', 'usa_view', 'usa_canadian_inflow_view',
                             'usa_subtotal_view', 'padd_canadian_inflow_view',
                             'usa_imports_view', 'padd_imports_view',
                             'usa_exports_view', 'padd_exports_view',
                             'usa_stocks_view', 'padd_stocks_view',
                             'foreign_export_destination',
                             'import_terminal', 'export_terminal')
       OR a.asset_id IN (
             'permian','bakken','eagle_ford','rest_of_l48',
             'padd1_production_view','padd2_production_view','padd3_production_view',
             'padd4_production_view','padd5_production_view',
             'spr_total')
"""), engine)
debug(f"Variables in scope (production + refining + SPR + import + export + boundary views): {len(vars_df)}")

outflow_counts = (
    vars_df[(vars_df['kind']=='physical') & (vars_df['variable_type']=='outflow') & (vars_df['node_class']=='production')]
    .groupby('node_id').size()
    .to_dict()
)

[14:24:57] Variables in scope (production + refining + SPR + import + export + boundary views): 1136


In [4]:
assignments = []

for vid, formula, inputs, notes in EXPLICIT_FORMULAS:
    assignments.append({"variable_id": vid, "formula": formula,
                        "formula_inputs": inputs, "notes": notes})

for vid, notes in LATENT_VARIABLES:
    assignments.append({"variable_id": vid, "formula": "latent()",
                        "formula_inputs": [], "notes": notes})

explicit_ids = {a["variable_id"] for a in assignments}

agg_children_by_node = {}
for vid, formula, inputs, _notes in EXPLICIT_FORMULAS:
    if formula == "sum" and vid.startswith("production__crude__"):
        node_id = vid[len("production__crude__"):]
        agg_children_by_node[node_id] = [v[len("production__crude__"):] for v in inputs]

phys_zero_types = ('consumption', 'inventory', 'balancing_item')

# 3a. Zeros on production-class physical nodes
for row in vars_df[(vars_df['node_class']=='production') & (vars_df['kind']=='physical') & (vars_df['variable_type'].isin(phys_zero_types))].itertuples():
    if row.variable_id in explicit_ids:
        continue
    if row.variable_type == 'balancing_item':
        # Per refactor 2026-05-11: no B=0 default on physical nodes. Latent.
        assignments.append({
            "variable_id": row.variable_id, "formula": "latent()", "formula_inputs": [],
            "notes": "Balancing item latent on production node (no closure observation)",
        })
    else:
        assignments.append({
            "variable_id": row.variable_id, "formula": "0", "formula_inputs": [],
            "notes": f"{row.variable_type} structurally zero on production node",
        })

# 3b. Production-node outflows: single = production; multi = latent()
phys_prod_nodes = vars_df[(vars_df['node_class']=='production') & (vars_df['kind']=='physical')]["node_id"].unique()
for nid in phys_prod_nodes:
    n_out = outflow_counts.get(nid, 0)
    if n_out == 0:
        continue
    out_rows = vars_df[(vars_df['variable_type']=='outflow') & (vars_df['node_id']==nid)]
    if n_out == 1:
        prod_var = f"production__crude__{nid}"
        for r in out_rows.itertuples():
            if r.variable_id in explicit_ids:
                continue
            assignments.append({
                "variable_id": r.variable_id, "formula": prod_var, "formula_inputs": [prod_var],
                "notes": "Single outflow carries full production under mass balance",
            })
    else:
        for r in out_rows.itertuples():
            if r.variable_id in explicit_ids:
                continue
            assignments.append({
                "variable_id": r.variable_id, "formula": "latent()", "formula_inputs": [],
                "notes": f"Latent: per-route split among {n_out} outflows from {nid} not observed",
            })

# 3c. Observational aggregates: handle ALL non-relational variables (P/C/I/B).
obs_var_types = ('production', 'consumption', 'inventory', 'balancing_item')
for row in vars_df[(vars_df['node_class']=='observational') & (vars_df['kind']=='abstract') & (vars_df['variable_type'].isin(obs_var_types))].itertuples():
    if row.variable_id in explicit_ids:
        continue
    children = agg_children_by_node.get(row.node_id, [])
    if children:
        inputs = [f"{row.variable_type}__crude__{c}" for c in children]
        assignments.append({
            "variable_id": row.variable_id, "formula": "sum", "formula_inputs": inputs,
            "notes": f"Aggregate {row.variable_type}: rollup of {len(children)} children",
        })
    else:
        assignments.append({
            "variable_id": row.variable_id, "formula": "0", "formula_inputs": [],
            "notes": f"Aggregate {row.variable_type}: no children defined; defaulted to 0",
        })

# Inter-PADD abstract outflows on padd_production_view / padd_refining_view that
# remain unbound (no formula and no TS) get filled with latent().
for row in vars_df[(vars_df['node_class']=='observational') & (vars_df['variable_type'].isin(['inflow','outflow']))].itertuples():
    if row.variable_id in explicit_ids:
        continue
    assignments.append({
        "variable_id": row.variable_id, "formula": "latent()", "formula_inputs": [],
        "notes": f"Aggregate {row.variable_type} (inter-PADD or abstract-flow): not observed under starter scope",
    })

# =========================================================================
# REFINING-SIDE PROGRAMMATIC
# =========================================================================
for row in vars_df[vars_df['node_subtype']=='refinery'].itertuples():
    if row.variable_id in explicit_ids:
        continue
    vt = row.variable_type
    if vt == 'production':
        assignments.append({"variable_id": row.variable_id, "formula": "0", "formula_inputs": [],
                            "notes": "Refineries do not produce crude (output is refined products, not modelled)"})
    elif vt == 'consumption':
        assignments.append({"variable_id": row.variable_id, "formula": "latent()", "formula_inputs": [],
                            "notes": "Refinery-level consumption latent — constrained by district-aggregate sum"})
    elif vt == 'inventory':
        # Default to latent (refineries DO hold inventory; per-facility data not public).
        # The PADD-aggregate MCRRSP{N} captures the sum; per-asset stays free for LP.
        assignments.append({"variable_id": row.variable_id, "formula": "latent()", "formula_inputs": [],
                            "notes": "Refinery crude inventory latent — per-asset data not public, aggregate via padd{N}_refinery_stocks"})
    elif vt == 'balancing_item':
        assignments.append({"variable_id": row.variable_id, "formula": "latent()", "formula_inputs": [],
                            "notes": "Refinery balancing item: latent (no per-asset closure observation)"})
    elif vt == 'inflow':
        assignments.append({"variable_id": row.variable_id, "formula": "latent()", "formula_inputs": [],
                            "notes": "Per-edge refinery inflow latent — per-pipeline split unobserved"})
    elif vt == 'outflow':
        assignments.append({"variable_id": row.variable_id, "formula": "0", "formula_inputs": [],
                            "notes": "Refineries do not outflow crude (refined products out of scope)"})

for row in vars_df[vars_df['node_subtype']=='refining_district_view'].itertuples():
    if row.variable_id in explicit_ids:
        continue
    vt = row.variable_type
    if vt == 'production':
        assignments.append({"variable_id": row.variable_id, "formula": "0", "formula_inputs": [],
                            "notes": "Refining-district aggregates do not produce crude"})
    elif vt == 'inventory':
        assignments.append({"variable_id": row.variable_id, "formula": "0", "formula_inputs": [],
                            "notes": "Refining-district inventory rollup default 0"})
    elif vt == 'balancing_item':
        assignments.append({"variable_id": row.variable_id, "formula": "latent()", "formula_inputs": [],
                            "notes": "Refining-district balancing item: latent"})

for row in vars_df[vars_df['node_subtype']=='refining_centre_view'].itertuples():
    if row.variable_id in explicit_ids:
        continue
    vt = row.variable_type
    if vt == 'production':
        assignments.append({"variable_id": row.variable_id, "formula": "0", "formula_inputs": [],
                            "notes": "PADD refining views do not produce crude"})
    elif vt == 'inventory':
        assignments.append({"variable_id": row.variable_id, "formula": "0", "formula_inputs": [],
                            "notes": "PADD refining inventory default 0"})
    elif vt == 'balancing_item':
        assignments.append({"variable_id": row.variable_id, "formula": "latent()", "formula_inputs": [],
                            "notes": "PADD refining-centre balancing item: latent (legacy node, retired by aggregate refactor)"})
    elif vt == 'inflow':
        assignments.append({"variable_id": row.variable_id, "formula": "latent()", "formula_inputs": [],
                            "notes": "PADD refining inflow (abstract-flow edge) latent"})
    elif vt == 'outflow':
        assignments.append({"variable_id": row.variable_id, "formula": "0", "formula_inputs": [],
                            "notes": "PADD refining views do not outflow crude (only consume)"})

# =========================================================================
# IMPORT-TERMINAL PROGRAMMATIC
# =========================================================================
for row in vars_df[vars_df['node_subtype']=='import_terminal'].itertuples():
    if row.variable_id in explicit_ids:
        continue
    vt = row.variable_type
    if vt == 'production':
        assignments.append({"variable_id": row.variable_id, "formula": "0", "formula_inputs": [],
                            "notes": "Import terminal default production 0 (transit point only)"})
    elif vt == 'consumption':
        assignments.append({"variable_id": row.variable_id, "formula": "0", "formula_inputs": [],
                            "notes": "Import terminal does not consume crude"})
    elif vt == 'inventory':
        assignments.append({"variable_id": row.variable_id, "formula": "0", "formula_inputs": [],
                            "notes": "Import terminal inventory not modelled (default 0)"})
    elif vt == 'balancing_item':
        assignments.append({"variable_id": row.variable_id, "formula": "latent()", "formula_inputs": [],
                            "notes": "Import terminal balancing item: latent"})
    elif vt == 'inflow':
        assignments.append({"variable_id": row.variable_id, "formula": "latent()", "formula_inputs": [],
                            "notes": "Per-edge import terminal inflow latent"})
    elif vt == 'outflow':
        assignments.append({"variable_id": row.variable_id, "formula": "latent()", "formula_inputs": [],
                            "notes": "Per-edge import terminal outflow to refineries latent"})

# =========================================================================
# SPR PROGRAMMATIC
# =========================================================================
SPR_SITES = ['spr_bryan_mound', 'spr_big_hill', 'spr_west_hackberry', 'spr_bayou_choctaw']
for row in vars_df[vars_df['node_subtype']=='spr_site'].itertuples():
    if row.variable_id in explicit_ids:
        continue
    vt = row.variable_type
    if vt == 'production':
        assignments.append({"variable_id": row.variable_id, "formula": "0", "formula_inputs": [],
                            "notes": "SPR sites do not produce crude"})
    elif vt == 'consumption':
        assignments.append({"variable_id": row.variable_id, "formula": "0", "formula_inputs": [],
                            "notes": "SPR sites do not consume crude (refineries do)"})
    elif vt == 'inventory':
        assignments.append({"variable_id": row.variable_id, "formula": "latent()", "formula_inputs": [],
                            "notes": "SPR site inventory: real quantity, no per-site EIA series loaded under starter"})
    elif vt == 'balancing_item':
        assignments.append({"variable_id": row.variable_id, "formula": "latent()", "formula_inputs": [],
                            "notes": "SPR site balancing item: latent"})
    elif vt == 'inflow':
        assignments.append({"variable_id": row.variable_id, "formula": "latent()", "formula_inputs": [],
                            "notes": "SPR fill: per-month per-site flow unobserved under starter scope"})
    elif vt == 'outflow':
        assignments.append({"variable_id": row.variable_id, "formula": "latent()", "formula_inputs": [],
                            "notes": "SPR release: per-month per-site flow unobserved under starter scope"})

for row in vars_df[vars_df['node_id']=='spr_total'].itertuples():
    if row.variable_id in explicit_ids:
        continue
    vt = row.variable_type
    if vt == 'inventory':
        inputs = [f"inventory__crude__{s}" for s in SPR_SITES]
        assignments.append({"variable_id": row.variable_id, "formula": "sum",
                            "formula_inputs": inputs,
                            "notes": "SPR total inventory = sum of 4 sites"})
    elif vt in ('production', 'consumption'):
        assignments.append({"variable_id": row.variable_id, "formula": "0", "formula_inputs": [],
                            "notes": f"SPR total {vt} default 0"})
    elif vt == 'balancing_item':
        assignments.append({"variable_id": row.variable_id, "formula": "latent()", "formula_inputs": [],
                            "notes": "SPR total balancing item: latent"})

# =========================================================================
# INFRASTRUCTURE PROGRAMMATIC — pipelines / hubs / origins / gathering / exports
# (added 2026-05-08): non-relational P/C/I/B default to 0 on transit nodes.
# Storage hubs and pipelines are an exception for INVENTORY: hub working
# storage and pipeline line-fill are real physical inventory, just not
# published per-asset by EIA. Mark them latent() so they are constrained
# jointly by PADD-level stock observations + node-level mass balance, rather
# than overwritten with a wrong 0.
# =========================================================================
infra_subtypes = ('pipeline', 'storage_terminal', 'export_terminal',
                  'origin_terminal', 'gathering', 'foreign_export_destination')
LATENT_INVENTORY_SUBTYPES = ('storage_terminal', 'pipeline')
infra_vars = pd.read_sql(text("""
    SELECT v.variable_id, v.variable_type, a.node_subtype
    FROM oil_network.variables v
    JOIN oil_network.assets a ON a.asset_id = v.node_id
    WHERE a.node_subtype = ANY(:subs)
      AND v.variable_type IN ('production','consumption','inventory','balancing_item')
"""), engine, params={"subs": list(infra_subtypes)})
for row in infra_vars.itertuples():
    if row.variable_id in explicit_ids:
        continue
    if row.variable_type == 'balancing_item':
        # Per design refactor 2026-05-11: no B=0 default. B is latent unless explicitly derived.
        assignments.append({
            "variable_id": row.variable_id, "formula": "latent()", "formula_inputs": [],
            "notes": f"Balancing item latent on {row.node_subtype} (no closure observation at this node)",
        })
    elif row.variable_type == 'inventory' and row.node_subtype in LATENT_INVENTORY_SUBTYPES:
        assignments.append({
            "variable_id": row.variable_id, "formula": "latent()", "formula_inputs": [],
            "notes": (f"Inventory latent on {row.node_subtype}: real working storage / line-fill, "
                      "no per-asset EIA observation. Constrained jointly by PADD-level stocks + mass balance."),
        })
    else:
        assignments.append({
            "variable_id": row.variable_id, "formula": "0", "formula_inputs": [],
            "notes": f"{row.variable_type} default 0 on {row.node_subtype} (transit node, no crude production/consumption)",
        })

# =========================================================================
# PASS-CLOSURE: every per-edge variable has an assignment (mirror pass)
# =========================================================================
all_outflows = pd.read_sql(text("""
    SELECT variable_id, node_id, related_node_id, commodity
    FROM oil_network.variables
    WHERE variable_type='outflow' AND related_node_id IS NOT NULL
"""), engine)
all_inflows = pd.read_sql(text("""
    SELECT variable_id, node_id, related_node_id, commodity
    FROM oil_network.variables
    WHERE variable_type='inflow' AND related_node_id IS NOT NULL
"""), engine)

# (i) Default-latent any outflow not already in `assignments`
already_assigned_ids = {a['variable_id'] for a in assignments}
n_default_outflow = 0
for row in all_outflows.itertuples():
    if row.variable_id in already_assigned_ids or row.variable_id in explicit_ids:
        continue
    assignments.append({
        "variable_id": row.variable_id, "formula": "latent()", "formula_inputs": [],
        "notes": f"Per-edge outflow on {row.node_id} -> {row.related_node_id} (no observation, default latent)",
    })
    n_default_outflow += 1

# (ii) Mirror every inflow to its paired outflow.
valid_outflow_ids = set(all_outflows['variable_id'])
n_mirrored = 0
for row in all_inflows.itertuples():
    if row.variable_id in explicit_ids:
        continue
    paired_outflow_id = f"outflow__{row.commodity}__{row.related_node_id}__{row.node_id}"
    if paired_outflow_id not in valid_outflow_ids:
        continue
    assignments.append({
        "variable_id": row.variable_id,
        "formula": paired_outflow_id,
        "formula_inputs": [paired_outflow_id],
        "notes": "Per-edge mass balance: inflow value = paired outflow value",
    })
    n_mirrored += 1

debug(f"Closure pass: defaulted {n_default_outflow} outflows to latent(); mirrored {n_mirrored} inflows to their paired outflow")

# Dedupe: keep the LAST assignment per variable_id. The mirror pass runs after
# any earlier latent() entries for the same inflow, so the mirror wins.
seen_assignments = {}
for a in assignments:
    seen_assignments[a['variable_id']] = a
assignments = list(seen_assignments.values())

debug(f"Total assignments to write: {len(assignments)}")
debug(f"  by formula type: {pd.Series([a['formula'] for a in assignments]).value_counts().head(8).to_dict()}")

[14:24:57] Closure pass: defaulted 251 outflows to latent(); mirrored 406 inflows to their paired outflow
[14:24:57] Total assignments to write: 1657


[14:24:57]   by formula type: {'latent()': 874, '0': 349, 'sum': 6, 'outflow__crude__padd3_view__foreign_export_destination': 2, 'production__crude__padd2_view - production__crude__bakken_nd - production__crude__oklahoma_conventional': 1, 'production__crude__padd3_view - production__crude__permian_tx - production__crude__permian_nm - production__crude__eagle_ford_tx - production__crude__gulf_of_america - production__crude__texas_other': 1, 'production__crude__padd4_view - production__crude__bakken_mt - production__crude__wyoming_conventional - production__crude__colorado_conventional - production__crude__montana_other': 1, 'production__crude__padd5_view - production__crude__california_conventional - production__crude__alaska_north_slope': 1}


## 3. Skip TS-bound variables, then UPSERT

In [5]:
ts_bound = pd.read_sql(text("""
    SELECT variable_id FROM oil_network.variable_assignments
    WHERE scenario_id = :s AND timeseries_id IS NOT NULL
"""), engine, params={"s": SCENARIO_ID})
ts_bound_ids = set(ts_bound['variable_id'])
debug(f"Skipping {len(ts_bound_ids)} TS-bound variables")

to_write = [a for a in assignments if a['variable_id'] not in ts_bound_ids]
debug(f"Writing {len(to_write)} formula-bound rows")

UPSERT = text("""
    INSERT INTO oil_network.variable_assignments
        (scenario_id, variable_id, effective_from, timeseries_id, formula, formula_inputs, notes)
    VALUES
        (:scenario_id, :variable_id, :effective_from, NULL, :formula, :formula_inputs, :notes)
    ON CONFLICT (scenario_id, variable_id, effective_from) DO UPDATE SET
        timeseries_id  = NULL,
        formula        = EXCLUDED.formula,
        formula_inputs = EXCLUDED.formula_inputs,
        notes          = EXCLUDED.notes
""")

rows = [
    {"scenario_id": SCENARIO_ID, "variable_id": a["variable_id"],
     "effective_from": EFFECTIVE_FROM, "formula": a["formula"],
     "formula_inputs": a["formula_inputs"], "notes": a["notes"]}
    for a in to_write
]

with engine.begin() as conn:
    conn.execute(UPSERT, rows)

n = pd.read_sql(text("""
    SELECT COUNT(*) FROM oil_network.variable_assignments
    WHERE scenario_id = :s AND formula IS NOT NULL
"""), engine, params={"s": SCENARIO_ID}).iloc[0, 0]
debug(f"variable_assignments (formula-bound, pre-cleanup): {n} rows")

# Cleanup: drop redundant + masking rows so variable_assignments stays sparse
# and v_effective_assignments fills these slots from node_type_default_formulas.
#
# Pass 1 — formula exactly matches the node_type default + no formula_inputs.
#          The override row was a tautology; the view layer would have produced
#          the same recipe.
#
# Pass 2 — formula='latent()' AND default='0'. The override would MASK the
#          structural-zero default that activates pass-through mass balance
#          (e.g. balancing_item__crude__permian_tx_gathering must default to 0
#          so the gathering closes ΣI = ΣO).
#
# Both passes preserve:
#   - TS-bound rows (timeseries_id NOT NULL)
#   - rows with non-empty formula_inputs ('sum', arithmetic, etc.)
#   - rows where formula='0' but default='latent()' (deliberate coverage-
#     contract decisions: refinery.inventory=0 collapses per-asset stocks to
#     PADD resolution; observational_aggregate basin views have S=0).
SPARSIFY_PASS1 = text("""
    WITH redundant AS (
        SELECT va.scenario_id, va.variable_id, va.effective_from
        FROM oil_network.variable_assignments va
        JOIN oil_network.variables v       ON v.variable_id = va.variable_id
        JOIN oil_network.nodes n           ON n.node_id = v.node_id
        JOIN oil_network.node_type_default_formulas ntdf
             ON ntdf.node_type     = n.node_type
            AND ntdf.variable_type = v.variable_type
            AND (ntdf.commodity IS NULL OR ntdf.commodity = v.commodity)
        WHERE va.scenario_id = :s
          AND va.timeseries_id IS NULL
          AND va.formula = ntdf.default_formula
          AND COALESCE(array_length(va.formula_inputs, 1), 0) = 0
    )
    DELETE FROM oil_network.variable_assignments va
    USING redundant r
    WHERE va.scenario_id    = r.scenario_id
      AND va.variable_id    = r.variable_id
      AND va.effective_from = r.effective_from
""")

SPARSIFY_PASS2 = text("""
    WITH masking AS (
        SELECT va.scenario_id, va.variable_id, va.effective_from
        FROM oil_network.variable_assignments va
        JOIN oil_network.variables v       ON v.variable_id = va.variable_id
        JOIN oil_network.nodes n           ON n.node_id = v.node_id
        JOIN oil_network.node_type_default_formulas ntdf
             ON ntdf.node_type     = n.node_type
            AND ntdf.variable_type = v.variable_type
            AND (ntdf.commodity IS NULL OR ntdf.commodity = v.commodity)
        WHERE va.scenario_id = :s
          AND va.timeseries_id IS NULL
          AND va.formula = 'latent()'
          AND ntdf.default_formula = '0'
          AND COALESCE(array_length(va.formula_inputs, 1), 0) = 0
    )
    DELETE FROM oil_network.variable_assignments va
    USING masking r
    WHERE va.scenario_id    = r.scenario_id
      AND va.variable_id    = r.variable_id
      AND va.effective_from = r.effective_from
""")

with engine.begin() as conn:
    r1 = conn.execute(SPARSIFY_PASS1, {"s": SCENARIO_ID})
    r2 = conn.execute(SPARSIFY_PASS2, {"s": SCENARIO_ID})
debug(f"Sparsified: pass1 deleted {r1.rowcount} redundant rows; pass2 deleted {r2.rowcount} masking latent() rows")

n_final = pd.read_sql(text("""
    SELECT COUNT(*) FROM oil_network.variable_assignments
    WHERE scenario_id = :s
"""), engine, params={"s": SCENARIO_ID}).iloc[0, 0]
debug(f"variable_assignments (post-sparsify): {n_final} rows; defaults fill the rest via v_effective_assignments")

[14:24:57] Skipping 0 TS-bound variables
[14:24:57] Writing 1657 formula-bound rows


[14:24:57] variable_assignments (formula-bound, pre-cleanup): 1657 rows


[14:24:59] Sparsified: pass1 deleted 717 redundant rows; pass2 deleted 88 masking latent() rows
[14:24:59] variable_assignments (post-sparsify): 852 rows; defaults fill the rest via v_effective_assignments


## 4. Verify

In [6]:
from IPython.display import display

print("-- Production-side coverage --")
display(pd.read_sql(text("""
    SELECT a.node_class, v.variable_type,
           COUNT(*) AS variables,
           COUNT(va.variable_id) AS assigned,
           COUNT(*) - COUNT(va.variable_id) AS unassigned
    FROM oil_network.variables v
    JOIN oil_network.nodes  n ON n.node_id = v.node_id
    JOIN oil_network.assets a ON a.asset_id = n.asset_id
    LEFT JOIN oil_network.variable_assignments va
        ON va.variable_id = v.variable_id AND va.scenario_id = :s
    WHERE a.node_class = 'production'
       OR (a.node_class = 'observational' AND a.asset_id IN (
             'permian','bakken','eagle_ford','rest_of_l48',
             'padd1_production_view','padd2_production_view','padd3_production_view',
             'padd4_production_view','padd5_production_view'))
    GROUP BY a.node_class, v.variable_type
    ORDER BY a.node_class, v.variable_type
"""), engine, params={"s": SCENARIO_ID}))

print("-- Totals by formula kind --")
display(pd.read_sql(text("""
    SELECT
        COUNT(*) AS total,
        COUNT(*) FILTER (WHERE timeseries_id IS NOT NULL) AS ts_bound,
        COUNT(*) FILTER (WHERE formula = 'latent()')      AS latent,
        COUNT(*) FILTER (WHERE formula = '0')             AS zeros,
        COUNT(*) FILTER (WHERE formula = 'sum')  AS sum_children,
        COUNT(*) FILTER (WHERE formula = 'sum')  AS sum_outflows,
        COUNT(*) FILTER (WHERE formula IS NOT NULL
                          AND formula NOT IN ('0','latent()','sum')) AS other_formula
    FROM oil_network.variable_assignments WHERE scenario_id = :s
"""), engine, params={"s": SCENARIO_ID}))

-- Production-side coverage --


,node_class,variable_type,variables,assigned,unassigned
0,observational,balancing_item,3,1,2
1,observational,consumption,3,1,2
2,observational,inventory,3,3,0
3,observational,production,3,3,0
4,production,balancing_item,19,0,19
5,production,consumption,19,0,19
6,production,inventory,19,0,19
7,production,outflow,89,89,0
8,production,production,19,10,9


-- Totals by formula kind --


,total,ts_bound,latent,zeros,sum_children,sum_outflows,other_formula
0,852,0,400,18,6,6,428
